In [ ]:
import os, re, math, json, time
from pathlib import Path
import numpy as np
import pandas as pd
import joblib

from xgboost import XGBRegressor
from sklearn.model_selection import KFold, train_test_split, ParameterGrid
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error,
    balanced_accuracy_score, classification_report
)


In [ ]:
from sklearn.model_selection import GroupKFold

In [ ]:
# === Rutas
TR_EMB = Path("/content/drive/MyDrive/TP1/data/deriv/embeddings/train_embeddings_v3.npz")
TE_EMB = Path("/content/drive/MyDrive/TP1/data/deriv/v3/embeddings/test_embeddings_v3.npz")

TRAIN_CSV = Path("/content/drive/MyDrive/TP1/data/deriv/v3/train_unified_aug.csv")
TEST_CSV  = Path("/content/drive/MyDrive/TP1/data/deriv/splits/test_unified.csv")

TARGET_NAME = "arousal"
positive_lang = "es"

In [ ]:
_base_aug_pat = re.compile(r"_aug\d+_.*$")
_norm_pat     = re.compile(r"_norm$")

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


# funciones

In [ ]:
def aggregate_clip(paths, preds, zrms=None, trim=0.10, alpha=0.0, eps=1e-6):
    """
    Agrega predicciones de ventanas a nivel clip con recorte 'trim'
    y ponderación por (zrms^alpha). Si alpha=0, ignora pesos.
    """
    paths = np.asarray(paths)
    preds = np.asarray(preds, dtype=np.float32)

    if zrms is None:
        z = np.ones_like(preds, dtype=np.float32)
    else:
        z = np.asarray(zrms, dtype=np.float32).reshape(-1)
        z = np.maximum(z, 0.0)  # no negativos

    groups = {}
    for i, (p, yhat) in enumerate(zip(paths, preds)):
        g = clip_group_from_path(p)
        lst = groups.setdefault(g, {"p": [], "w": []})
        lst["p"].append(float(yhat))
        lst["w"].append(float(z[i]))

    yhat_clip = {}
    for g, d in groups.items():
        p = np.asarray(d["p"], dtype=np.float64)
        w = np.asarray(d["w"], dtype=np.float64)

        # pesos por alpha (si alpha=0 → todos 1)
        if alpha != 0.0:
            w = np.power(w + eps, float(alpha))
        else:
            w = np.ones_like(w, dtype=np.float64)

        # recorte por predicción
        order = np.argsort(p)
        k = int(np.floor(trim * len(p)))
        if k > 0 and (len(p) - 2 * k) >= 1:
            p = p[order][k:-k]
            w = w[order][k:-k]

        s = w.sum()
        yhat_clip[g] = float(p.mean() if s <= 0 else np.dot(p, w) / s)

    return yhat_clip

In [ ]:
def clip_group_from_path(p: str) -> str:
    s = Path(p).stem
    s = _base_aug_pat.sub("", s)
    s = _norm_pat.sub("", s)
    parent = Path(p).parent.name
    return f"{parent}/{s}"

def ccc(y, yhat, eps=1e-12):
    y = np.asarray(y, float); yhat = np.asarray(yhat, float)
    mu_y, mu_h = y.mean(), yhat.mean()
    var_y, var_h = y.var(), yhat.var()
    cov = np.mean((y-mu_y)*(yhat-mu_h))
    return float((2*cov)/(var_y + var_h + (mu_y-mu_h)**2 + eps))

def lang_by_clip_from_paths(paths, langs):
    by = {}
    for p, l in zip(paths, langs):
        g = clip_group_from_path(p)
        by.setdefault(g, []).append(l)
    out = {}
    for g, L in by.items():
        u, c = np.unique(np.asarray(L, "U"), return_counts=True)
        out[g] = u[int(np.argmax(c))]
    return out

def lang_weights(langs_fold, target_prior={"es":0.5,"qu":0.5}, cap=4.0, smoothing=1e-3):
    Ls = np.asarray(langs_fold, "U")
    uniq, cnt = np.unique(Ls, return_counts=True)
    L = max(1, len(uniq))
    total = cnt.sum()
    obs = {u:(cnt[i]+smoothing)/(total+smoothing*L) for i,u in enumerate(uniq)}
    ratio = {u: target_prior.get(u, 0.0)/max(obs.get(u,1e-12),1e-12) for u in uniq}
    w = np.array([ratio.get(l,1.0) for l in Ls], np.float32)
    w = np.clip(w, 1.0/cap, cap)
    return w * (len(w)/w.sum())

In [ ]:
def _find_col(df, keys):
    for k in keys:
        if k in df.columns: return k
    raise KeyError(f"No encuentro {keys} en columnas: {df.columns.tolist()}")

In [ ]:
def y_from_paths(paths, m):
    y = np.empty(len(paths), np.float32)
    miss = 0
    for i,p in enumerate(paths):
        if p in m:
            y[i] = float(m[p])
        else:
            # fallback por clip_id base
            g = clip_group_from_path(p)
            # buscar alguna fila con ese clip base
            # (si tu CSV de test no tiene aug, este fallback ayuda)
            hit = df_te[df_te[col_path].apply(lambda s: clip_group_from_path(s)==g)]
            if len(hit):
                y[i] = float(hit.iloc[0][col_val])
            else:
                y[i] = np.nan; miss += 1
    if miss:
        print(f"[Aviso] {miss} paths sin match → NaN")
    return y

In [ ]:
def fit_iso_per_lang(ypc, ytc, langs):
    out = {}
    for lg in np.unique(langs):
        m = (langs==lg)
        if m.sum() < 10:
            out[lg] = None
        else:
            out[lg] = IsotonicRegression(y_min=1.0, y_max=5.0, out_of_bounds="clip").fit(ypc[m], ytc[m])
    return out

def apply_iso_per_lang(ypc, langs, cal):
    y = np.empty_like(ypc, float)
    for lg in np.unique(langs):
        m = (langs==lg)
        if cal.get(lg) is None:
            y[m] = ypc[m]
        else:
            y[m] = cal[lg].transform(ypc[m])
    return y

In [ ]:
def _metrics(y, p, tag):
    c = ccc(y, p)
    mae = float(mean_absolute_error(y, p))
    rmse = float(math.sqrt(mean_squared_error(y, p)))
    print(f"=== TEST {tag} ===  MAE={mae:.4f}  RMSE={rmse:.4f}  CCC={c:.4f}")


In [ ]:
def compute_ccc(y_true, y_pred):
    """
    Robust CCC implementation with better numerical stability
    """
    y_true = np.asarray(y_true, dtype=np.float64).flatten()
    y_pred = np.asarray(y_pred, dtype=np.float64).flatten()

    # Remove invalid values
    mask = np.isfinite(y_true) & np.isfinite(y_pred)
    y_true = y_true[mask]
    y_pred = y_pred[mask]

    if len(y_true) < 2:
        return 0.0

    # Pearson correlation
    cor = np.corrcoef(y_true, y_pred)[0, 1]

    # Means and standard deviations
    mean_true = np.mean(y_true)
    mean_pred = np.mean(y_pred)
    std_true = np.std(y_true, ddof=1)
    std_pred = np.std(y_pred, ddof=1)

    # CCC formula using Pearson correlation
    numerator = 2 * cor * std_true * std_pred
    denominator = std_true**2 + std_pred**2 + (mean_true - mean_pred)**2

    if denominator == 0:
        return 0.0

    ccc = numerator / denominator

    return float(ccc)

In [ ]:
from sklearn.model_selection import ParameterGrid
import numpy as np

def optimize_aggregation(yp_val_w, path_val, y_val_w, zrms_val):
    """Optimiza parámetros de agregación en validación"""

    # Grid de búsqueda
    param_grid = {
        'trim': [0.05, 0.08, 0.10, 0.12, 0.15, 0.18, 0.20],
        'alpha': [0.0, 0.05, 0.10, 0.15, 0.20, 0.25, 0.30]
    }

    best_ccc = -1
    best_params = None
    results = []

    for params in ParameterGrid(param_grid):
        # Agregar predicciones
        yhat_clip = aggregate_clip(
            path_val, yp_val_w,
            zrms=zrms_val.ravel(),
            trim=params['trim'],
            alpha=params['alpha']
        )

        # Ground truth por clip
        ytrue_clip = {}
        for p, yt in zip(path_val, y_val_w):
            g = clip_group_from_path(p)
            ytrue_clip.setdefault(g, []).append(float(yt))
        ytrue_clip = {g: float(np.mean(v)) for g, v in ytrue_clip.items()}

        # Alinear
        keys = sorted(set(yhat_clip) & set(ytrue_clip))
        ypc = np.array([yhat_clip[k] for k in keys], np.float32)
        ytc = np.array([ytrue_clip[k] for k in keys], np.float32)

        # Calcular CCC
        ccc = compute_ccc(ypc, ytc)
        results.append({'trim': params['trim'], 'alpha': params['alpha'], 'ccc': ccc})

        if ccc > best_ccc:
            best_ccc = ccc
            best_params = params

    print(f"Best aggregation params: trim={best_params['trim']}, alpha={best_params['alpha']}, CCC={best_ccc:.4f}")
    return best_params, results

In [ ]:
def train_xgb(X_train, y_train, sample_weight, X_val=None, y_val=None, params=None):
    params = params or {
        "objective": "reg:squarederror",
        "learning_rate": 0.05,
        "n_estimators": 2000,
        "grow_policy": "lossguide",
        "max_depth": 0,
        "max_leaves": 32,
        "min_child_weight": 10.0,
        "subsample": 0.75,
        "colsample_bytree": 0.80,
        "colsample_bynode": 0.70,
        "gamma": 0.20,
        "reg_alpha": 1.0,
        "reg_lambda": 35.0,
        "max_bin": 128,
        "tree_method": "hist",
        "random_state": 42,
        "n_jobs": -1,
        "early_stopping_rounds": 50,
        "eval_metric": "rmse"
    }

    mdl = XGBRegressor(**params)
    if X_val is not None and y_val is not None:
        mdl.fit(X_train, y_train, sample_weight=sample_weight,
                eval_set=[(X_val, y_val)], verbose=100)
    else:
        mdl.fit(X_train, y_train, sample_weight=sample_weight, verbose=False)
    return mdl


In [ ]:
def evaluate_model(mdl, X_tr, y_tr, path_tr, zrms_tr, lang_tr,
                   X_te, y_te, path_te, zrms_te, lang_te,
                   trim=0.10, alpha=0.0):

    # Agregacion (train)
    yp_tr = mdl.predict(X_tr)
    yhat_tr = aggregate_clip(path_tr, yp_tr, zrms_tr, trim, alpha)
    ytrue_tr = {}
    for p, yt in zip(path_tr, y_tr):
        g = clip_group_from_path(p)
        ytrue_tr.setdefault(g, []).append(float(yt))
    ytrue_tr = {g: np.mean(v) for g, v in ytrue_tr.items()}
    keys_tr = sorted(set(yhat_tr) & set(ytrue_tr))
    ypc_tr, ytc_tr = np.array([yhat_tr[k] for k in keys_tr]), np.array([ytrue_tr[k] for k in keys_tr])

    iso_global = IsotonicRegression(y_min=1.0, y_max=5.0, out_of_bounds="clip").fit(ypc_tr, ytc_tr)
    langs_tr = np.array([lang_by_clip_from_paths(path_tr, lang_tr)[k] for k in keys_tr])
    cal_lang = fit_iso_per_lang(ypc_tr, ytc_tr, langs_tr)

    # Agregacion (test)
    yp_te = mdl.predict(X_te)
    yhat_te = aggregate_clip(path_te, yp_te, zrms_te, trim, alpha)
    ytrue_te = {}
    for p, yt in zip(path_te, y_te):
        g = clip_group_from_path(p)
        ytrue_te.setdefault(g, []).append(float(yt))
    ytrue_te = {g: np.mean(v) for g, v in ytrue_te.items()}
    keys_te = sorted(set(yhat_te) & set(ytrue_te))
    ypc_te, ytc_te = np.array([yhat_te[k] for k in keys_te]), np.array([ytrue_te[k] for k in keys_te])
    langs_te = np.array([lang_by_clip_from_paths(path_te, lang_te)[k] for k in keys_te])

    # Evalua
    _metrics(ytc_te, ypc_te, "(raw)")
    _metrics(ytc_te, iso_global.transform(ypc_te), "(calibrado GLOBAL)")
    _metrics(ytc_te, apply_iso_per_lang(ypc_te, langs_te, cal_lang), "(calibrado POR IDIOMA)")

    # Clasificacion (3-level)
    ytc_cls = np.digitize(ytc_te, bins=[2.5, 3.5], right=True)
    ypc_cls = np.digitize(apply_iso_per_lang(ypc_te, langs_te, cal_lang), bins=[2.5, 3.5], right=True)
    print("\nBalanced Acc (H/M/L):", balanced_accuracy_score(ytc_cls, ypc_cls))
    print(classification_report(ytc_cls, ypc_cls,
          target_names=["Bajo", "Medio", "Alto"], digits=4))

    return iso_global, cal_lang

In [ ]:
def save_model_bundle(model, iso_global, cal_lang, export_path, target_name="valence",
                      positive_lang="es", agg_kwargs=None, xgb_params=None, n_features=None, mu_rms=None, sd_rms=None):
    export_path = Path(export_path)
    export_path.mkdir(parents=True, exist_ok=True)

    joblib.dump(model, export_path / "xgb_model.joblib", compress=3)
    joblib.dump(iso_global, export_path / "iso_global.joblib", compress=3)
    joblib.dump(cal_lang, export_path / "iso_per_lang.joblib", compress=3)

    meta = {
        "saved_at": time.strftime("%Y-%m-%d %H:%M:%S"),
        "target": target_name,
        "positive_lang": positive_lang,
        "agg_kwargs": agg_kwargs or {"trim": 0.10, "alpha": 0.0},
        "xgb_params": xgb_params,
        "n_features": int(n_features or 0),
        "mu_rms": mu_rms,
        "sd_rms": sd_rms
    }
    with open(export_path / "meta.json", "w") as f:
        json.dump(meta, f, indent=2)

    print("Guardado en:", export_path)

In [ ]:
def concordance_ccc(y_true, y_pred):
    y_true = np.asarray(y_true, np.float64)
    y_pred = np.asarray(y_pred, np.float64)
    mu_t, mu_p = y_true.mean(), y_pred.mean()
    vt, vp = y_true.var(), y_pred.var()
    cov = np.mean((y_true - mu_t) * (y_pred - mu_p))
    return (2 * cov) / (vt + vp + (mu_t - mu_p) ** 2 + 1e-8)

In [ ]:
def agg_clip(paths, preds, zrms):
    return aggregate_clip(paths, preds, zrms=zrms.ravel(), trim=0.10, alpha=0.0)

In [ ]:
def to_bins(y):
    return np.digitize(y, bins=[2.5, 3.5], right=True)

# carga

In [ ]:
Dt = np.load(TR_EMB, allow_pickle=True)
Xe = Dt["Xw"]

path_tr = Dt["path"].astype("U")
zrms_tr = Dt["zrms"].astype(np.float32)
lang_tr = Dt["lang"].astype("U")
Xw_tr   = Xe.astype(np.float32)

De = np.load(TE_EMB, allow_pickle=True)
Xe = De["Xw"]
path_te = De["path"].astype("U")
zrms_te = De["zrms"].astype(np.float32)
lang_te = De["lang"].astype("U")
Xw_te   = Xe.astype(np.float32)

# columnas extra
is_es_tr = (lang_tr == "es").astype(np.float32)[:, None]
is_es_te = (lang_te == "es").astype(np.float32)[:, None]

Xw_tr = np.concatenate([Xw_tr, is_es_tr, zrms_tr[:, None]], axis=1).astype(np.float32)
Xw_te = np.concatenate([Xw_te, is_es_te, zrms_te[:, None]], axis=1).astype(np.float32)

print("Train windows:", Xw_tr.shape, " Test windows:", Xw_te.shape)


Train windows: (132621, 2050)  Test windows: (15732, 2050)


In [ ]:
WIN_TR_NPZ = Path("/content/drive/MyDrive/TP1/data/deriv/v3/windows/windows_train_v3.npz")

W = np.load(WIN_TR_NPZ, allow_pickle=True)

In [ ]:
mu_rms = float(W["mu_rms"])
sd_rms = float(W["sd_rms"])

In [ ]:
rms_tr = zrms_tr * sd_rms + mu_rms
rms_te = zrms_te * sd_rms + mu_rms

# 2) log1p del RMS
log_tr = np.log1p(np.clip(rms_tr, 1e-9, None))
log_te = np.log1p(np.clip(rms_te, 1e-9, None))

# 3) log con stats del train
mu_log = float(log_tr.mean())
sd_log = float(log_tr.std(ddof=1) + 1e-9)
zrms_log_tr = (log_tr - mu_log) / sd_log
zrms_log_te = (log_te - mu_log) / sd_log

# 4) columna extra
Xw_tr = np.concatenate([Xw_tr, zrms_log_tr[:, None]], axis=1).astype(np.float32)
Xw_te = np.concatenate([Xw_te, zrms_log_te[:, None]], axis=1).astype(np.float32)

print("Nuevas shapes:", Xw_tr.shape, Xw_te.shape)

Nuevas shapes: (132621, 2051) (15732, 2051)


In [ ]:
df_tr = pd.read_csv(TRAIN_CSV)
df_te = pd.read_csv(TEST_CSV)

In [ ]:
df_tr['path_norm'].fillna(df_tr['path'], inplace=True)

/tmp/ipython-input-3513165660.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_tr['path_norm'].fillna(df_tr['path'], inplace=True)


In [ ]:
df_tr['is_aug'].fillna(False, inplace=True)

/tmp/ipython-input-3729766301.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_tr['is_aug'].fillna(False, inplace=True)
/tmp/ipython-input-3729766301.py:1: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_tr['is_aug'].fillna(False, inplace=True)


In [ ]:
df_tr['sr'].fillna('', inplace=True)
df_tr['aug_ops'].fillna('', inplace=True)

/tmp/ipython-input-2401029001.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_tr['sr'].fillna('', inplace=True)
/tmp/ipython-input-2401029001.py:1: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df_tr['sr'].fillna('', inplace=True)
/tmp/ipython-input-2401029001.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series thro

In [ ]:
col_path = _find_col(df_tr, ["path_norm", "path"])

In [ ]:
col_val  = _find_col(df_tr, ["arousal","Arousal"])

In [ ]:
map_tr   = df_tr.set_index(col_path)[col_val].astype(float).to_dict()

In [ ]:
col_path = _find_col(df_te, ["path_norm", "path"])
map_te   = df_te.set_index(col_path)[col_val].astype(float).to_dict()

In [ ]:
y_tr_w = y_from_paths(path_tr, map_tr)
y_te_w = y_from_paths(path_te, map_te)

In [ ]:
mtr = np.isfinite(y_tr_w); mte = np.isfinite(y_te_w)
Xw_tr, path_tr, zrms_tr, lang_tr, y_tr_w = Xw_tr[mtr], path_tr[mtr], zrms_tr[mtr], lang_tr[mtr], y_tr_w[mtr]
Xw_te, path_te, zrms_te, lang_te, y_te_w = Xw_te[mte], path_te[mte], zrms_te[mte], lang_te[mte], y_te_w[mte]

print("Targets TRAIN (min/mean/max):", float(y_tr_w.min()), float(y_tr_w.mean()), float(y_tr_w.max()))
print("Targets TEST  (min/mean/max):", float(y_te_w.min()), float(y_te_w.mean()), float(y_te_w.max()))

Targets TRAIN (min/mean/max): 1.0 3.0483312606811523 5.0
Targets TEST  (min/mean/max): 1.0 3.102180242538452 5.0


In [ ]:
g_tr = np.array([clip_group_from_path(p) for p in path_tr])

# pesos por clip
ug, cnt = np.unique(g_tr, return_counts=True)
cnt_map = {g:c for g,c in zip(ug, cnt)}
sw_clip = np.array([1.0/cnt_map[g] for g in g_tr], dtype=np.float32)
sw_clip *= (len(sw_clip) / sw_clip.sum())  # media≈1

# prior de idioma (ajusta si tienes otros tags)
prior = {"es": 0.5, "qu": 0.5}

# pesos por idioma
w_lang = lang_weights(lang_tr, target_prior=prior)

# combina, clip y renormaliza
W_full = np.clip(sw_clip * w_lang, 1e-3, 20.0).astype(np.float32)
W_full *= (len(W_full) / W_full.sum())  # media≈1

print("W_full: min=%.3f  p50=%.3f  p90=%.3f  max=%.3f  mean=%.3f" % (
    float(W_full.min()), float(np.median(W_full)), float(np.quantile(W_full, 0.90)),
    float(W_full.max()), float(W_full.mean())
))


W_full: min=0.393  p50=0.787  p90=1.574  max=7.869  mean=1.000


# Pruebas

## Primero

In [ ]:
(X_train, X_val,
 y_train, y_val,
 W_train, W_val,
 path_train, path_val,
 zrms_train, zrms_val,
 lang_train, lang_val) = train_test_split(
    Xw_tr, y_tr_w, W_full, path_tr, zrms_tr, lang_tr,
    test_size=0.15,
    random_state=42,
    shuffle=True
)

print(f"Training samples: {len(X_train)}")
print(f"Validation samples: {len(X_val)}")
print(f"Weight stats - Train: min={W_train.min():.3f}, mean={W_train.mean():.3f}, max={W_train.max():.3f}")
print(f"Weight stats - Val: min={W_val.min():.3f}, mean={W_val.mean():.3f}, max={W_val.max():.3f}")

Training samples: 112727
Validation samples: 19894
Weight stats - Train: min=0.393, mean=0.999, max=7.869
Weight stats - Val: min=0.393, mean=1.005, max=7.869


In [ ]:
best_params = {
    "objective": "reg:squarederror",
    "learning_rate": 0.05,
    "n_estimators": 2000,
    "grow_policy": "lossguide",
    "max_depth": 0,
    "max_leaves": 32,
    "min_child_weight": 10.0,
    "subsample": 0.75,
    "colsample_bytree": 0.80,
    "colsample_bynode": 0.70,
    "gamma": 0.20,
    "reg_alpha": 1.0,
    "reg_lambda": 35.0,
    "max_bin": 128,
    "tree_method": "hist",
    "random_state": 42,
    "n_jobs": -1,
    "device": "cuda",
    "early_stopping_rounds": 50,
    "eval_metric": "rmse"
}

In [ ]:
mdl = train_xgb(
    X_train, y_train, W_train,
    X_val=X_val, y_val=y_val,
    params=best_params
)

[0]	validation_0-rmse:0.83275
[50]	validation_0-rmse:0.66605
[100]	validation_0-rmse:0.63561
[150]	validation_0-rmse:0.62002
[200]	validation_0-rmse:0.60955
[250]	validation_0-rmse:0.60116
[300]	validation_0-rmse:0.59450
[350]	validation_0-rmse:0.58896
[400]	validation_0-rmse:0.58458
[450]	validation_0-rmse:0.58060
[500]	validation_0-rmse:0.57722
[550]	validation_0-rmse:0.57411
[600]	validation_0-rmse:0.57154
[650]	validation_0-rmse:0.56914
[700]	validation_0-rmse:0.56684
[750]	validation_0-rmse:0.56487
[800]	validation_0-rmse:0.56297
[850]	validation_0-rmse:0.56108
[900]	validation_0-rmse:0.55946
[950]	validation_0-rmse:0.55793
[1000]	validation_0-rmse:0.55649
[1050]	validation_0-rmse:0.55489
[1100]	validation_0-rmse:0.55323
[1150]	validation_0-rmse:0.55178
[1200]	validation_0-rmse:0.55050
[1250]	validation_0-rmse:0.54931
[1300]	validation_0-rmse:0.54803
[1350]	validation_0-rmse:0.54669
[1400]	validation_0-rmse:0.54565
[1450]	validation_0-rmse:0.54468
[1500]	validation_0-rmse:0.54364


XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=0.7, colsample_bytree=0.8,
             device='cuda', early_stopping_rounds=50, enable_categorical=False,
             eval_metric='rmse', feature_types=None, feature_weights=None,
             gamma=0.2, grow_policy='lossguide', importance_type=None,
             interaction_constraints=None, learning_rate=0.05, max_bin=128,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=0, max_leaves=32,
             min_child_weight=10.0, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=2000, n_jobs=-1,
             num_parallel_tree=None, ...)

In [ ]:
iso_global, cal_lang = evaluate_model(
    mdl,
    Xw_tr, y_tr_w, path_tr, zrms_tr, lang_tr,
    Xw_te, y_te_w, path_te, zrms_te, lang_te
)


=== TRAIN (clip, raw) ===  MAE=0.2803 RMSE=0.3576 CCC=0.8906
=== TEST (raw) ===  MAE=0.4780  RMSE=0.5867  CCC=0.7057
=== TEST (calibrado GLOBAL) ===  MAE=0.4722  RMSE=0.5927  CCC=0.7335
=== TEST (calibrado POR-IDIOMA) ===  MAE=0.4665  RMSE=0.5959  CCC=0.7308


In [ ]:
ytc_cls = np.digitize(ytc_te, bins=[2.5, 3.5], right=True)
ypc_cls = np.digitize(apply_iso_per_lang(ypc_te, langs_clip_te, cal_lang), bins=[2.5, 3.5], right=True)
print("\nBalanced Acc (H/M/L):", balanced_accuracy_score(ytc_cls, ypc_cls))
print(classification_report(ytc_cls, ypc_cls,
      target_names=["Bajo","Medio","Alto"], digits=4))


Balanced Acc (H/M/L): 0.6382580474516323
              precision    recall  f1-score   support

        Bajo     0.6837    0.4607    0.5504       699
       Medio     0.5616    0.7105    0.6273       988
        Alto     0.7793    0.7436    0.7610       741

    accuracy                         0.6487      2428
   macro avg     0.6749    0.6383    0.6463      2428
weighted avg     0.6632    0.6487    0.6460      2428



In [ ]:
agg_kwargs = {"trim":0.10,"alpha":0.0}

In [ ]:
save_model_bundle(
    mdl, iso_global, cal_lang,
    "/content/drive/MyDrive/TP1/modelos/arousal/v2/primero",
    target_name="arousal",
    positive_lang="es",
    xgb_params=best_params,
    n_features=Xw_tr.shape[1]
)

Guardado en: /content/drive/MyDrive/TP1/modelos/arousal/v2/primero


## Optuna

In [ ]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.9/400.9 kB 15.5 MB/s eta 0:00:00


In [ ]:
import optuna
from sklearn.metrics import mean_squared_error

def ccc2(y_true, y_pred):
    y_true_mean = np.mean(y_true)
    y_pred_mean = np.mean(y_pred)
    covariance = np.mean((y_true - y_true_mean) * (y_pred - y_pred_mean))
    true_var = np.var(y_true)
    pred_var = np.var(y_pred)
    return (2 * covariance) / (true_var + pred_var + (y_true_mean - y_pred_mean) ** 2 + 1e-8)

def objective(trial):
    params = {
        "objective": "reg:squarederror",
        "tree_method": "hist",
        "device": "cuda",
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1),
        "n_estimators": 2000,
        "max_depth": 0,  # grow_policy con max_leaves
        "grow_policy": "lossguide",
        "max_leaves": trial.suggest_int("max_leaves", 16, 64),
        "min_child_weight": trial.suggest_float("min_child_weight", 5.0, 20.0),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "colsample_bynode": trial.suggest_float("colsample_bynode", 0.5, 1.0),
        "gamma": trial.suggest_float("gamma", 0.0, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 0.0, 10.0),
        "reg_lambda": trial.suggest_float("reg_lambda", 1.0, 100.0),
        "max_bin": trial.suggest_int("max_bin", 64, 256),
        "early_stopping_rounds": 50,
        "eval_metric": "rmse",
        "random_state": 42,
        "n_jobs": -1,
    }

    model = XGBRegressor(**params)
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        verbose=150
    )

    preds = model.predict(X_val)
    ccc = ccc2(y_val, preds)
    print(f"CCC: {ccc}")
    return 1 - ccc  # minimizar (1 - CCC) para maximizar CCC

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=50)

# Mejor resultado
print("Mejores hiperparámetros:", study.best_params)
print("Mejor CCC en validación:", 1 - study.best_value)


[I 2025-10-07 20:16:46,858] A new study created in memory with name: no-name-87ebb52f-14c9-4172-8f87-ca4110f92ad6


[0]	validation_0-rmse:0.83707
[150]	validation_0-rmse:0.68171
[300]	validation_0-rmse:0.64037
[450]	validation_0-rmse:0.61960
[600]	validation_0-rmse:0.60599
[750]	validation_0-rmse:0.59604
[900]	validation_0-rmse:0.58809
[1050]	validation_0-rmse:0.58166
[1200]	validation_0-rmse:0.57621
[1350]	validation_0-rmse:0.57144
[1500]	validation_0-rmse:0.56717
[1650]	validation_0-rmse:0.56345
[1800]	validation_0-rmse:0.56004
[1950]	validation_0-rmse:0.55698
[1999]	validation_0-rmse:0.55611


[I 2025-10-07 20:21:44,176] Trial 0 finished with value: 0.3009788990020752 and parameters: {'learning_rate': 0.01034436815273985, 'max_leaves': 60, 'min_child_weight': 6.872636976887654, 'subsample': 0.6644503391825667, 'colsample_bytree': 0.5432653072176752, 'colsample_bynode': 0.6781034112137942, 'gamma': 0.719263423680531, 'reg_alpha': 9.16076250483527, 'reg_lambda': 40.292001514100825, 'max_bin': 237}. Best is trial 0 with value: 0.3009788990020752.


CCC: 0.6990211009979248
[0]	validation_0-rmse:0.83337
[150]	validation_0-rmse:0.62190
[300]	validation_0-rmse:0.59161
[450]	validation_0-rmse:0.57530
[600]	validation_0-rmse:0.56454
[750]	validation_0-rmse:0.55647
[900]	validation_0-rmse:0.55012
[1050]	validation_0-rmse:0.54506
[1200]	validation_0-rmse:0.54086
[1350]	validation_0-rmse:0.53723
[1500]	validation_0-rmse:0.53389
[1650]	validation_0-rmse:0.53082
[1800]	validation_0-rmse:0.52786
[1950]	validation_0-rmse:0.52522
[1999]	validation_0-rmse:0.52438


[I 2025-10-07 20:24:39,043] Trial 1 finished with value: 0.2563050389289856 and parameters: {'learning_rate': 0.03507342215288042, 'max_leaves': 48, 'min_child_weight': 12.828377767913233, 'subsample': 0.6485091569024116, 'colsample_bytree': 0.536650916779126, 'colsample_bynode': 0.570444703380641, 'gamma': 0.47246317521356296, 'reg_alpha': 3.44511969905696, 'reg_lambda': 77.8872163020419, 'max_bin': 81}. Best is trial 1 with value: 0.2563050389289856.


...

CCC: 0.7632312178611755
[0]	validation_0-rmse:0.82712
[150]	validation_0-rmse:0.58743
[300]	validation_0-rmse:0.56067
[450]	validation_0-rmse:0.54778
[600]	validation_0-rmse:0.53913
[750]	validation_0-rmse:0.53233
[900]	validation_0-rmse:0.52695
[1050]	validation_0-rmse:0.52250
[1200]	validation_0-rmse:0.51858
[1350]	validation_0-rmse:0.51524
[1500]	validation_0-rmse:0.51210
[1650]	validation_0-rmse:0.50932
[1800]	validation_0-rmse:0.50700
[1950]	validation_0-rmse:0.50471
[1999]	validation_0-rmse:0.50402


[I 2025-10-07 23:40:14,356] Trial 48 finished with value: 0.22943496704101562 and parameters: {'learning_rate': 0.07065102063213163, 'max_leaves': 52, 'min_child_weight': 16.507190794553388, 'subsample': 0.6474096756723879, 'colsample_bytree': 0.7562867234373424, 'colsample_bynode': 0.6770995680106987, 'gamma': 0.26431176931231315, 'reg_alpha': 3.117765765441618, 'reg_lambda': 73.57201656814823, 'max_bin': 245}. Best is trial 44 with value: 0.22008824348449707.


CCC: 0.7705650329589844
[0]	validation_0-rmse:0.82518
[150]	validation_0-rmse:0.58078
[300]	validation_0-rmse:0.55479
[450]	validation_0-rmse:0.54181
[600]	validation_0-rmse:0.53311
[750]	validation_0-rmse:0.52652
[900]	validation_0-rmse:0.52108
[1050]	validation_0-rmse:0.51658
[1200]	validation_0-rmse:0.51255
[1350]	validation_0-rmse:0.50925
[1500]	validation_0-rmse:0.50632
[1650]	validation_0-rmse:0.50387
[1800]	validation_0-rmse:0.50164
[1950]	validation_0-rmse:0.50020
[1999]	validation_0-rmse:0.49991


[I 2025-10-07 23:43:59,273] Trial 49 finished with value: 0.2251874804496765 and parameters: {'learning_rate': 0.07909571783728973, 'max_leaves': 62, 'min_child_weight': 14.734624589814013, 'subsample': 0.794941534522488, 'colsample_bytree': 0.7312796743887464, 'colsample_bynode': 0.8417147807782299, 'gamma': 0.23020224467552358, 'reg_alpha': 5.413874805148702, 'reg_lambda': 79.97508349810656, 'max_bin': 112}. Best is trial 44 with value: 0.22008824348449707.


CCC: 0.7748125195503235
Mejores hiperparámetros: {'learning_rate': 0.09456364697246283, 'max_leaves': 64, 'min_child_weight': 11.067093244776393, 'subsample': 0.6421371772487362, 'colsample_bytree': 0.6776813173232048, 'colsample_bynode': 0.8049407417792415, 'gamma': 0.13711395942225835, 'reg_alpha': 5.5721972528548696, 'reg_lambda': 65.47548348174428, 'max_bin': 234}
Mejor CCC en validación: 0.7799117565155029


In [ ]:
best_params = {'learning_rate': 0.09456364697246283,
               'max_leaves': 64,
               'min_child_weight': 11.067093244776393,
               'subsample': 0.6421371772487362,
               'colsample_bytree': 0.6776813173232048,
               'colsample_bynode': 0.8049407417792415,
               'gamma': 0.13711395942225835,
               'reg_alpha': 5.5721972528548696,
               'reg_lambda': 65.47548348174428,
               'max_bin': 234,
               'device':'cuda',
               "objective": "reg:squarederror",
               "tree_method": "hist",
               "n_estimators": 2000,
                "max_depth": 0,  # grow_policy con max_leaves
                "grow_policy": "lossguide",
               "early_stopping_rounds": 50,
                "eval_metric": "rmse",
                "random_state": 42,
                "n_jobs": -1,}

In [ ]:
# Train model
mdl = XGBRegressor(**best_params)
mdl.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    sample_weight=W_train,
    verbose=50
)

[0]	validation_0-rmse:0.82327
[50]	validation_0-rmse:0.62788
[100]	validation_0-rmse:0.60008
[150]	validation_0-rmse:0.58413
[200]	validation_0-rmse:0.57423
[250]	validation_0-rmse:0.56673
[300]	validation_0-rmse:0.56089
[350]	validation_0-rmse:0.55642
[400]	validation_0-rmse:0.55232
[450]	validation_0-rmse:0.54913
[500]	validation_0-rmse:0.54585
[550]	validation_0-rmse:0.54302
[600]	validation_0-rmse:0.54029
[650]	validation_0-rmse:0.53813
[700]	validation_0-rmse:0.53615
[750]	validation_0-rmse:0.53416
[800]	validation_0-rmse:0.53276
[850]	validation_0-rmse:0.53131
[900]	validation_0-rmse:0.52970
[950]	validation_0-rmse:0.52803
[1000]	validation_0-rmse:0.52674
[1050]	validation_0-rmse:0.52538
[1100]	validation_0-rmse:0.52402
[1150]	validation_0-rmse:0.52271
[1200]	validation_0-rmse:0.52140
[1250]	validation_0-rmse:0.52014
[1300]	validation_0-rmse:0.51886
[1350]	validation_0-rmse:0.51781
[1400]	validation_0-rmse:0.51677
[1450]	validation_0-rmse:0.51593
[1500]	validation_0-rmse:0.51504


XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=0.8049407417792415,
             colsample_bytree=0.6776813173232048, device='cuda',
             early_stopping_rounds=50, enable_categorical=False,
             eval_metric='rmse', feature_types=None, feature_weights=None,
             gamma=0.13711395942225835, grow_policy='lossguide',
             importance_type=None, interaction_constraints=None,
             learning_rate=0.09456364697246283, max_bin=234,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=0, max_leaves=64,
             min_child_weight=11.067093244776393, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=2000,
             n_jobs=-1, num_parallel_tree=None, ...)

In [ ]:
print(f"\nBest iteration: {mdl.best_iteration}")
print(f"Best RMSE: {mdl.best_score:.4f}")


Best iteration: 1999
Best RMSE: 0.5083


In [ ]:
yp_tr_w = mdl.predict(Xw_tr).astype(np.float32)
yhat_tr_clip = aggregate_clip(path_tr, yp_tr_w, zrms=zrms_tr.ravel(), trim=0.10, alpha=0.0)
ytrue_tr_clip = {}
for p, yt in zip(path_tr, y_tr_w):
    g = clip_group_from_path(p)
    ytrue_tr_clip.setdefault(g, []).append(float(yt))
ytrue_tr_clip = {g: float(np.mean(v)) for g,v in ytrue_tr_clip.items()}
keys_tr = sorted(set(yhat_tr_clip) & set(ytrue_tr_clip))
ypc_tr = np.array([yhat_tr_clip[k]  for k in keys_tr], np.float32)
ytc_tr = np.array([ytrue_tr_clip[k] for k in keys_tr], np.float32)

iso_global = IsotonicRegression(y_min=1.0, y_max=5.0, out_of_bounds="clip").fit(ypc_tr, ytc_tr)

clip2lang_tr = lang_by_clip_from_paths(path_tr, lang_tr)
langs_clip_tr = np.array([clip2lang_tr[k] for k in keys_tr])

In [ ]:
cal_lang = fit_iso_per_lang(ypc_tr, ytc_tr, langs_clip_tr)

In [ ]:
yp_te_w = mdl.predict(Xw_te).astype(np.float32)
yhat_te_clip = aggregate_clip(path_te, yp_te_w, zrms=zrms_te.ravel(), trim=0.10, alpha=0.0)
ytrue_te_clip = {}
for p, yt in zip(path_te, y_te_w):
    g = clip_group_from_path(p)
    ytrue_te_clip.setdefault(g, []).append(float(yt))
ytrue_te_clip = {g: float(np.mean(v)) for g,v in ytrue_te_clip.items()}

keys_te = sorted(set(yhat_te_clip) & set(ytrue_te_clip))
ypc_te = np.array([yhat_te_clip[k]  for k in keys_te], np.float32)
ytc_te = np.array([ytrue_te_clip[k] for k in keys_te], np.float32)

clip2lang_te = lang_by_clip_from_paths(path_te, lang_te)
langs_clip_te = np.array([clip2lang_te[k] for k in keys_te])


In [ ]:
print("\n=== TRAIN (clip, raw) ===  MAE=%.4f RMSE=%.4f CCC=%.4f" % (
    float(mean_absolute_error(ytc_tr, ypc_tr)),
    float(math.sqrt(mean_squared_error(ytc_tr, ypc_tr))),
    ccc(ytc_tr, ypc_tr)
))
_metrics(ytc_te, ypc_te, "(raw)")
_metrics(ytc_te, iso_global.transform(ypc_te), "(calibrado GLOBAL)")
_metrics(ytc_te, apply_iso_per_lang(ypc_te, langs_clip_te, cal_lang), "(calibrado POR-IDIOMA)")


=== TRAIN (clip, raw) ===  MAE=0.1469 RMSE=0.2029 CCC=0.9681
=== TEST (raw) ===  MAE=0.4735  RMSE=0.5812  CCC=0.7157
=== TEST (calibrado GLOBAL) ===  MAE=0.4648  RMSE=0.5830  CCC=0.7347
=== TEST (calibrado POR-IDIOMA) ===  MAE=0.4576  RMSE=0.5939  CCC=0.7284


segundo mejor(mejores resultados ccc):

In [ ]:
best_params = {'learning_rate': 0.09989730695216865,
               'max_leaves': 63,
               'min_child_weight': 11.503343698077279,
               'subsample': 0.6154070311066155,
               'colsample_bytree': 0.6652044024893341,
               'colsample_bynode': 0.788136481728704,
               'gamma': 0.13549424394028625,
               'reg_alpha': 7.701025741480394,
               'reg_lambda': 47.55490206989469,
               'max_bin': 255,
               'device':'cuda',
               "objective": "reg:squarederror",
               "tree_method": "hist",
               "n_estimators": 2000,
                "max_depth": 0,  # grow_policy con max_leaves
                "grow_policy": "lossguide",
               "early_stopping_rounds": 50,
                "eval_metric": "rmse",
                "random_state": 42,
                "n_jobs": -1,}

In [ ]:
# Train model
mdl = XGBRegressor(**best_params)
mdl.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    sample_weight=W_train,
    verbose=50
)

[0]	validation_0-rmse:0.82369
[50]	validation_0-rmse:0.62550
[100]	validation_0-rmse:0.59817
[150]	validation_0-rmse:0.58389
[200]	validation_0-rmse:0.57351
[250]	validation_0-rmse:0.56614
[300]	validation_0-rmse:0.56003
[350]	validation_0-rmse:0.55568
[400]	validation_0-rmse:0.55172
[450]	validation_0-rmse:0.54810
[500]	validation_0-rmse:0.54520
[550]	validation_0-rmse:0.54277
[600]	validation_0-rmse:0.54021
[650]	validation_0-rmse:0.53797
[700]	validation_0-rmse:0.53622
[750]	validation_0-rmse:0.53404
[800]	validation_0-rmse:0.53211
[850]	validation_0-rmse:0.53045
[900]	validation_0-rmse:0.52895
[950]	validation_0-rmse:0.52752
[1000]	validation_0-rmse:0.52623
[1050]	validation_0-rmse:0.52484
[1100]	validation_0-rmse:0.52360
[1150]	validation_0-rmse:0.52223
[1200]	validation_0-rmse:0.52089
[1250]	validation_0-rmse:0.51981
[1300]	validation_0-rmse:0.51892
[1350]	validation_0-rmse:0.51802
[1400]	validation_0-rmse:0.51692
[1450]	validation_0-rmse:0.51585
[1500]	validation_0-rmse:0.51480


XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=0.788136481728704,
             colsample_bytree=0.6652044024893341, device='cuda',
             early_stopping_rounds=50, enable_categorical=False,
             eval_metric='rmse', feature_types=None, feature_weights=None,
             gamma=0.13549424394028625, grow_policy='lossguide',
             importance_type=None, interaction_constraints=None,
             learning_rate=0.09989730695216865, max_bin=255,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=0, max_leaves=63,
             min_child_weight=11.503343698077279, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=2000,
             n_jobs=-1, num_parallel_tree=None, ...)

In [ ]:
print(f"\nBest iteration: {mdl.best_iteration}")
print(f"Best RMSE: {mdl.best_score:.4f}")


Best iteration: 1999
Best RMSE: 0.5083


In [ ]:
yp_tr_w = mdl.predict(Xw_tr).astype(np.float32)
yhat_tr_clip = aggregate_clip(path_tr, yp_tr_w, zrms=zrms_tr.ravel(), trim=0.10, alpha=0.0)
ytrue_tr_clip = {}
for p, yt in zip(path_tr, y_tr_w):
    g = clip_group_from_path(p)
    ytrue_tr_clip.setdefault(g, []).append(float(yt))
ytrue_tr_clip = {g: float(np.mean(v)) for g,v in ytrue_tr_clip.items()}
keys_tr = sorted(set(yhat_tr_clip) & set(ytrue_tr_clip))
ypc_tr = np.array([yhat_tr_clip[k]  for k in keys_tr], np.float32)
ytc_tr = np.array([ytrue_tr_clip[k] for k in keys_tr], np.float32)

iso_global = IsotonicRegression(y_min=1.0, y_max=5.0, out_of_bounds="clip").fit(ypc_tr, ytc_tr)

clip2lang_tr = lang_by_clip_from_paths(path_tr, lang_tr)
langs_clip_tr = np.array([clip2lang_tr[k] for k in keys_tr])

In [ ]:
cal_lang = fit_iso_per_lang(ypc_tr, ytc_tr, langs_clip_tr)

In [ ]:
yp_te_w = mdl.predict(Xw_te).astype(np.float32)
yhat_te_clip = aggregate_clip(path_te, yp_te_w, zrms=zrms_te.ravel(), trim=0.10, alpha=0.0)
ytrue_te_clip = {}
for p, yt in zip(path_te, y_te_w):
    g = clip_group_from_path(p)
    ytrue_te_clip.setdefault(g, []).append(float(yt))
ytrue_te_clip = {g: float(np.mean(v)) for g,v in ytrue_te_clip.items()}

keys_te = sorted(set(yhat_te_clip) & set(ytrue_te_clip))
ypc_te = np.array([yhat_te_clip[k]  for k in keys_te], np.float32)
ytc_te = np.array([ytrue_te_clip[k] for k in keys_te], np.float32)

clip2lang_te = lang_by_clip_from_paths(path_te, lang_te)
langs_clip_te = np.array([clip2lang_te[k] for k in keys_te])


In [ ]:
print("\n=== TRAIN (clip, raw) ===  MAE=%.4f RMSE=%.4f CCC=%.4f" % (
    float(mean_absolute_error(ytc_tr, ypc_tr)),
    float(math.sqrt(mean_squared_error(ytc_tr, ypc_tr))),
    ccc(ytc_tr, ypc_tr)
))
_metrics(ytc_te, ypc_te, "(raw)")
_metrics(ytc_te, iso_global.transform(ypc_te), "(calibrado GLOBAL)")
_metrics(ytc_te, apply_iso_per_lang(ypc_te, langs_clip_te, cal_lang), "(calibrado POR-IDIOMA)")


=== TRAIN (clip, raw) ===  MAE=0.1482 RMSE=0.2047 CCC=0.9675
=== TEST (raw) ===  MAE=0.4707  RMSE=0.5795  CCC=0.7183
=== TEST (calibrado GLOBAL) ===  MAE=0.4624  RMSE=0.5813  CCC=0.7371
=== TEST (calibrado POR-IDIOMA) ===  MAE=0.4551  RMSE=0.5932  CCC=0.7289


In [ ]:
ytc_cls = np.digitize(ytc_te, bins=[2.5, 3.5], right=True)
ypc_cls = np.digitize(iso_global.transform(ypc_te), bins=[2.5, 3.5], right=True)
print("\nBalanced Acc (H/M/L):", balanced_accuracy_score(ytc_cls, ypc_cls))
print(classification_report(ytc_cls, ypc_cls,
      target_names=["Bajo","Medio","Alto"], digits=4))


Balanced Acc (H/M/L): 0.6431932836382802
              precision    recall  f1-score   support

        Bajo     0.6944    0.4778    0.5661       699
       Medio     0.5644    0.7095    0.6287       988
        Alto     0.7801    0.7422    0.7607       741

    accuracy                         0.6528      2428
   macro avg     0.6796    0.6432    0.6518      2428
weighted avg     0.6677    0.6528    0.6510      2428



In [ ]:
import json, joblib, time
EXPORT = Path("/content/drive/MyDrive/TP1/modelos/arousal/v2/segundo")
EXPORT.mkdir(parents=True, exist_ok=True)

# XGBRegressor en joblib (rápido de recargar)
joblib.dump(mdl, EXPORT/"xgb_model.joblib", compress=3)

# Calibradores
joblib.dump(iso_global, EXPORT/"iso_global.joblib", compress=3)
joblib.dump(cal_lang,   EXPORT/"iso_per_lang.joblib", compress=3)

# Metadatos/config mínima
meta = {
    "saved_at": time.strftime("%Y-%m-%d %H:%M:%S"),
    "target": globals().get("TARGET_NAME", "arousal"),
    "positive_lang": positive_lang,
    "agg_kwargs": dict(agg_kwargs),
    "xgb_params": best_params,
    "n_features": int(Xw_tr.shape[1]),
}

if "mu_rms" in globals(): meta["mu_rms"] = float(mu_rms)
if "sd_rms" in globals(): meta["sd_rms"] = float(sd_rms)

with open(EXPORT/"meta.json", "w") as f:
    json.dump(meta, f, indent=2)

print("Guardado en:", EXPORT)


Guardado en: /content/drive/MyDrive/TP1/modelos/arousal/v2/segundo


## validacion por folds (ver generalizacion del modelo)

In [ ]:
best_params = {'learning_rate': 0.09989730695216865,
               'max_leaves': 63,
               'min_child_weight': 11.503343698077279,
               'subsample': 0.6154070311066155,
               'colsample_bytree': 0.6652044024893341,
               'colsample_bynode': 0.788136481728704,
               'gamma': 0.13549424394028625,
               'reg_alpha': 7.701025741480394,
               'reg_lambda': 47.55490206989469,
               'max_bin': 255,
               'device':'cuda',
               "objective": "reg:squarederror",
               "tree_method": "hist",
               "n_estimators": 2000,
                "max_depth": 0,  # grow_policy con max_leaves
                "grow_policy": "lossguide",
               "early_stopping_rounds": 50,
                "eval_metric": "rmse",
                "random_state": 42,
                "n_jobs": -1,}

In [ ]:
def concordance_ccc(y_true, y_pred):
    mean_true = np.mean(y_true)
    mean_pred = np.mean(y_pred)
    cov = np.mean((y_true - mean_true) * (y_pred - mean_pred))
    std_true = np.std(y_true)
    std_pred = np.std(y_pred)
    return (2 * cov) / (std_true**2 + std_pred**2 + (mean_true - mean_pred)**2 + 1e-8)

In [ ]:
(X_train, X_val,
 y_train, y_val,
 W_train, W_val,
 path_train, path_val,
 zrms_train, zrms_val,
 lang_train, lang_val) = train_test_split(
    Xw_tr, y_tr_w, W_full, path_tr, zrms_tr, lang_tr,
    test_size=0.15,
    random_state=42,
    shuffle=True
)

In [ ]:
ccc_scores = []
kf = KFold(n_splits=5, shuffle=True, random_state=42)

for train_idx, val_idx in kf.split(Xw_tr):
    # Train/val splits con arrays
    X_tr, X_val = Xw_tr[train_idx], Xw_tr[val_idx]
    y_tr, y_val = y_tr_w[train_idx], y_tr_w[val_idx]
    w_tr = W_full[train_idx]

    path_tr_fold = path_tr[train_idx]
    path_val_fold = path_tr[val_idx]
    zrms_tr_fold = zrms_tr[train_idx]
    zrms_val_fold = zrms_tr[val_idx]
    lang_tr_fold = lang_tr[train_idx]
    lang_val_fold = lang_tr[val_idx]

    # --- Modelo base ---
    model = XGBRegressor(**best_params)
    model.fit(X_tr, y_tr, sample_weight=w_tr,
              eval_set=[(X_val, y_val)],
              verbose=150)

    # --- Predicciones entrenamiento (para calibrar) ---
    yp_tr = model.predict(X_tr).astype(np.float32)
    yhat_tr_clip = aggregate_clip(path_tr_fold, yp_tr, zrms=zrms_tr_fold.ravel(), trim=0.10, alpha=0.0)

    ytrue_tr_clip = {}
    for p, yt in zip(path_tr_fold, y_tr):
        g = clip_group_from_path(p)
        ytrue_tr_clip.setdefault(g, []).append(float(yt))
    ytrue_tr_clip = {g: float(np.mean(v)) for g, v in ytrue_tr_clip.items()}

    keys_tr = sorted(set(yhat_tr_clip) & set(ytrue_tr_clip))
    ypc_tr = np.array([yhat_tr_clip[k] for k in keys_tr], np.float32)
    ytc_tr = np.array([ytrue_tr_clip[k] for k in keys_tr], np.float32)
    clip2lang_tr = lang_by_clip_from_paths(path_tr_fold, lang_tr_fold)
    langs_clip_tr = np.array([clip2lang_tr[k] for k in keys_tr])

    iso_global = IsotonicRegression(y_min=1.0, y_max=5.0, out_of_bounds="clip").fit(ypc_tr, ytc_tr)

    # --- Predicciones validación ---
    yp_val = model.predict(X_val).astype(np.float32)
    yhat_val_clip = aggregate_clip(path_val_fold, yp_val, zrms=zrms_val_fold.ravel(), trim=0.10, alpha=0.0)

    ytrue_val_clip = {}
    for p, yt in zip(path_val_fold, y_val):
        g = clip_group_from_path(p)
        ytrue_val_clip.setdefault(g, []).append(float(yt))
    ytrue_val_clip = {g: float(np.mean(v)) for g, v in ytrue_val_clip.items()}

    keys_val = sorted(set(yhat_val_clip) & set(ytrue_val_clip))
    ypc_val = np.array([yhat_val_clip[k] for k in keys_val], np.float32)
    ytc_val = np.array([ytrue_val_clip[k] for k in keys_val], np.float32)
    clip2lang_val = lang_by_clip_from_paths(path_val_fold, lang_val_fold)
    langs_clip_val = np.array([clip2lang_val[k] for k in keys_val])

    # --- Calibrar predicciones ---
    ypc_val_cal = iso_global.predict(ypc_val)

    # --- CCC calibrado ---
    ccc = concordance_ccc(ytc_val, ypc_val_cal)
    ccc_scores.append(ccc)

print(f"\n=== CCC por fold calibrado === Mean= {np.mean(ccc_scores):.4f}, Std= {np.std(ccc_scores):.4f}")

[0]	validation_0-rmse:0.82255
[150]	validation_0-rmse:0.58168
[300]	validation_0-rmse:0.55930
[450]	validation_0-rmse:0.54808
[600]	validation_0-rmse:0.53997
[750]	validation_0-rmse:0.53389
[900]	validation_0-rmse:0.52892
[1050]	validation_0-rmse:0.52485
[1200]	validation_0-rmse:0.52127
[1350]	validation_0-rmse:0.51856
[1500]	validation_0-rmse:0.51596
[1650]	validation_0-rmse:0.51419
[1800]	validation_0-rmse:0.51226
[1950]	validation_0-rmse:0.51084
[1999]	validation_0-rmse:0.51057


/usr/local/lib/python3.12/dist-packages/xgboost/core.py:729: UserWarning: [01:43:04] WARNING: /workspace/src/common/error_msg.cc:58: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


[0]	validation_0-rmse:0.82474
[150]	validation_0-rmse:0.58516
[300]	validation_0-rmse:0.56187
[450]	validation_0-rmse:0.54820
[600]	validation_0-rmse:0.53951
[750]	validation_0-rmse:0.53333
[900]	validation_0-rmse:0.52803
[1050]	validation_0-rmse:0.52367
[1200]	validation_0-rmse:0.51983
[1350]	validation_0-rmse:0.51656
[1500]	validation_0-rmse:0.51391
[1650]	validation_0-rmse:0.51184
[1800]	validation_0-rmse:0.50977
[1950]	validation_0-rmse:0.50819
[1999]	validation_0-rmse:0.50782
[0]	validation_0-rmse:0.82830
[150]	validation_0-rmse:0.58398
[300]	validation_0-rmse:0.56000
[450]	validation_0-rmse:0.54800
[600]	validation_0-rmse:0.53990
[750]	validation_0-rmse:0.53377
[900]	validation_0-rmse:0.52856
[1050]	validation_0-rmse:0.52428
[1200]	validation_0-rmse:0.52087
[1350]	validation_0-rmse:0.51800
[1500]	validation_0-rmse:0.51520
[1650]	validation_0-rmse:0.51273
[1800]	validation_0-rmse:0.51056
[1950]	validation_0-rmse:0.50909
[1999]	validation_0-rmse:0.50869
[0]	validation_0-rmse:0.8282

# nuevas agregaciones(mfcc_mean, mfcc_std, pitch_features)

In [ ]:
ACOUSTIC_FEAT_DIR = Path("/content/drive/MyDrive/TP1/data/deriv/v3/acoustic_features")
WIN_TR_NPZ = Path("/content/drive/MyDrive/TP1/data/deriv/v3/windows/windows_train_v3.npz")

In [ ]:
Dt = np.load(TR_EMB, allow_pickle=True)
Xw_tr, path_tr, zrms_tr, lang_tr = Dt["Xw"], Dt["path"], Dt["zrms"], Dt["lang"]

De = np.load(TE_EMB, allow_pickle=True)
Xw_te, path_te, zrms_te, lang_te = De["Xw"], De["path"], De["zrms"], De["lang"]

In [ ]:
AF_tr = np.load(ACOUSTIC_FEAT_DIR / "train_acoustic_features_v3.npz")
acoustic_feats_tr = AF_tr["acoustic_features"]
AF_te = np.load(ACOUSTIC_FEAT_DIR / "test_acoustic_features_v3.npz")
acoustic_feats_te = AF_te["acoustic_features"]

# --- 2. Preparación de todas las características ---

# Normaliza las nuevas características acústicas
mu_ac = np.mean(acoustic_feats_tr, axis=0)
sd_ac = np.std(acoustic_feats_tr, axis=0) + 1e-8
acoustic_feats_tr_norm = (acoustic_feats_tr - mu_ac) / sd_ac
acoustic_feats_te_norm = (acoustic_feats_te - mu_ac) / sd_ac

# Prepara la columna 'is_es'
is_es_tr = (lang_tr == "es").astype(np.float32)[:, None]
is_es_te = (lang_te == "es").astype(np.float32)[:, None]

In [ ]:
W = np.load(WIN_TR_NPZ, allow_pickle=True)
mu_rms, sd_rms = float(W["mu_rms"]), float(W["sd_rms"])
rms_tr = zrms_tr * sd_rms + mu_rms
log_tr = np.log1p(np.clip(rms_tr, 1e-9, None))
mu_log, sd_log = float(log_tr.mean()), float(log_tr.std(ddof=1) + 1e-9)
zrms_log_tr = ((log_tr - mu_log) / sd_log)[:, None] # Asegura que sea una columna
rms_te = zrms_te * sd_rms + mu_rms
log_te = np.log1p(np.clip(rms_te, 1e-9, None))
zrms_log_te = ((log_te - mu_log) / sd_log)[:, None] # Asegura que sea una columna

In [ ]:
Xw_tr = np.concatenate([
    Xw_tr.astype(np.float32),      # Embeddings wav2vec
    is_es_tr,                      # Es español?
    zrms_tr[:, None],              # Z-score de RMS
    zrms_log_tr,                   # Z-score del log de RMS
    acoustic_feats_tr_norm
], axis=1)

Xw_te = np.concatenate([
    Xw_te.astype(np.float32),      # Embeddings wav2vec
    is_es_te,                      # Es español?
    zrms_te[:, None],              # Z-score de RMS
    zrms_log_te,                   # Z-score del log de RMS
    acoustic_feats_te_norm
], axis=1)

print(f"Shape final de entrenamiento: {Xw_tr.shape}")
print(f"Shape final de test: {Xw_te.shape}")

Shape final de entrenamiento: (132621, 2081)
Shape final de test: (15732, 2081)


In [ ]:
df_tr = pd.read_csv(TRAIN_CSV)
df_te = pd.read_csv(TEST_CSV)

In [ ]:
df_tr['path_norm'].fillna(df_tr['path'], inplace=True)

In [ ]:
df_tr['is_aug'].fillna(False, inplace=True)

In [ ]:
df_tr['sr'].fillna('', inplace=True)
df_tr['aug_ops'].fillna('', inplace=True)

In [ ]:
df_tr

,path,corpus,lang,speaker_id,clip_id,valence,arousal,dominance,path_norm,sr,is_aug,aug_ops
0,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,globalv1,es,Audio1,Audio1_10-02-03-04,2.0,3.0,4.0,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,,False,
1,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,globalv1,es,Audio1,Audio1_104-03-02-04,3.0,2.0,4.0,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,,False,
2,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,globalv1,es,Audio1,Audio1_1-02-03-03,2.0,3.0,3.0,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,,False,
3,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,globalv1,es,Audio1,Audio1_105-03-02-04,3.0,2.0,4.0,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,,False,
4,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,globalv1,es,Audio1,Audio1_102-03-04-05,3.0,4.0,5.0,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,,False,
...,...,...,...,...,...,...,...,...,...,...,...,...
20146,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,globalv1,es,Audio9,Audio9_83-03-03-04,3.0,3.0,4.0,/content/drive/MyDrive/TP1/data/deriv/v3/train...,16000.0,True,ts1.013
20147,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,globalv1,es,Audio9,Audio9_99-02-02-04,2.0,2.0,4.0,/content/drive/MyDrive/TP1/data/deriv/v3/train...,16000.0,True,ns32.2
20148,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,globalv1,es,Audio9,Audio9_99-02-02-04,2.0,2.0,4.0,/content/drive/MyDrive/TP1/data/deriv/v3/train...,16000.0,True,ns33.2
20149,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,globalv1,es,Audio9,Audio9_86-03-03-04,3.0,3.0,4.0,/content/drive/MyDrive/TP1/data/deriv/v3/train...,16000.0,True,rvb0.12s_0.11


In [ ]:
col_path = _find_col(df_tr, ["path_norm", "path"])

In [ ]:
col_val  = _find_col(df_tr, ["arousal","Arousal"])

In [ ]:
map_tr   = df_tr.set_index(col_path)[col_val].astype(float).to_dict()

In [ ]:
col_path = _find_col(df_te, ["path_norm", "path"])
map_te   = df_te.set_index(col_path)[col_val].astype(float).to_dict()

In [ ]:
y_tr_w = y_from_paths(path_tr, map_tr)
y_te_w = y_from_paths(path_te, map_te)

In [ ]:
mtr = np.isfinite(y_tr_w); mte = np.isfinite(y_te_w)
Xw_tr, path_tr, zrms_tr, lang_tr, y_tr_w = Xw_tr[mtr], path_tr[mtr], zrms_tr[mtr], lang_tr[mtr], y_tr_w[mtr]
Xw_te, path_te, zrms_te, lang_te, y_te_w = Xw_te[mte], path_te[mte], zrms_te[mte], lang_te[mte], y_te_w[mte]

print("Targets TRAIN (min/mean/max):", float(y_tr_w.min()), float(y_tr_w.mean()), float(y_tr_w.max()))
print("Targets TEST  (min/mean/max):", float(y_te_w.min()), float(y_te_w.mean()), float(y_te_w.max()))

Targets TRAIN (min/mean/max): 1.0 3.0483312606811523 5.0
Targets TEST  (min/mean/max): 1.0 3.102180242538452 5.0


In [ ]:
g_tr = np.array([clip_group_from_path(p) for p in path_tr])

# pesos por clip
ug, cnt = np.unique(g_tr, return_counts=True)
cnt_map = {g:c for g,c in zip(ug, cnt)}
sw_clip = np.array([1.0/cnt_map[g] for g in g_tr], dtype=np.float32)
sw_clip *= (len(sw_clip) / sw_clip.sum())  # media≈1

# prior de idioma
prior = {"es": 0.5, "qu": 0.5}

# pesos por idioma
w_lang = lang_weights(lang_tr, target_prior=prior)

# combina, clip y renormaliza
W_full = np.clip(sw_clip * w_lang, 1e-3, 20.0).astype(np.float32)
W_full *= (len(W_full) / W_full.sum())  # media≈1

print("W_full: min=%.3f  p50=%.3f  p90=%.3f  max=%.3f  mean=%.3f" % (
    float(W_full.min()), float(np.median(W_full)), float(np.quantile(W_full, 0.90)),
    float(W_full.max()), float(W_full.mean())
))


W_full: min=0.393  p50=0.787  p90=1.574  max=7.869  mean=1.000


## Prueba con mejores resultados de Optuna

con mejores parms de optuna (**menor** ccc)

In [ ]:
best_params = {'learning_rate': 0.09989730695216865,
               'max_leaves': 63,
               'min_child_weight': 11.503343698077279,
               'subsample': 0.6154070311066155,
               'colsample_bytree': 0.6652044024893341,
               'colsample_bynode': 0.788136481728704,
               'gamma': 0.13549424394028625,
               'reg_alpha': 7.701025741480394,
               'reg_lambda': 47.55490206989469,
               'max_bin': 255,
               'device':'cuda',
               "objective": "reg:squarederror",
               "tree_method": "hist",
               "n_estimators": 2000,
                "max_depth": 0,  # grow_policy con max_leaves
                "grow_policy": "lossguide",
               "early_stopping_rounds": 50,
                "eval_metric": "rmse",
                "random_state": 42,
                "n_jobs": -1,}

In [ ]:
mdl = train_xgb(
    X_train, y_train, W_train,
    X_val=X_val, y_val=y_val,
    params=best_params
)

[0]	validation_0-rmse:0.81826
[50]	validation_0-rmse:0.60095
[100]	validation_0-rmse:0.57785
[150]	validation_0-rmse:0.56414
[200]	validation_0-rmse:0.55545
[250]	validation_0-rmse:0.54914
[300]	validation_0-rmse:0.54414
[350]	validation_0-rmse:0.53980
[400]	validation_0-rmse:0.53599
[450]	validation_0-rmse:0.53310
[500]	validation_0-rmse:0.53017
[550]	validation_0-rmse:0.52762
[600]	validation_0-rmse:0.52537
[650]	validation_0-rmse:0.52322
[700]	validation_0-rmse:0.52116
[750]	validation_0-rmse:0.51958
[800]	validation_0-rmse:0.51809
[850]	validation_0-rmse:0.51633
[900]	validation_0-rmse:0.51487
[950]	validation_0-rmse:0.51346
[1000]	validation_0-rmse:0.51199
[1050]	validation_0-rmse:0.51059
[1100]	validation_0-rmse:0.50920
[1150]	validation_0-rmse:0.50817
[1200]	validation_0-rmse:0.50708
[1250]	validation_0-rmse:0.50611
[1300]	validation_0-rmse:0.50512
[1350]	validation_0-rmse:0.50423
[1400]	validation_0-rmse:0.50330
[1450]	validation_0-rmse:0.50232
[1500]	validation_0-rmse:0.50158


XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=0.788136481728704,
             colsample_bytree=0.6652044024893341, device='cuda',
             early_stopping_rounds=50, enable_categorical=False,
             eval_metric='rmse', feature_types=None, feature_weights=None,
             gamma=0.13549424394028625, grow_policy='lossguide',
             importance_type=None, interaction_constraints=None,
             learning_rate=0.09989730695216865, max_bin=255,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=0, max_leaves=63,
             min_child_weight=11.503343698077279, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=2000,
             n_jobs=-1, num_parallel_tree=None, ...)

In [ ]:
iso_global, cal_lang = evaluate_model(
    mdl,
    Xw_tr, y_tr_w, path_tr, zrms_tr, lang_tr,
    Xw_te, y_te_w, path_te, zrms_te, lang_te
)


=== TRAIN (clip, raw) ===  MAE=0.1456 RMSE=0.2002 CCC=0.9691
=== TEST (raw) ===  MAE=0.4644  RMSE=0.5716  CCC=0.7301
=== TEST (calibrado GLOBAL) ===  MAE=0.4580  RMSE=0.5741  CCC=0.7464
=== TEST (calibrado POR-IDIOMA) ===  MAE=0.4494  RMSE=0.5843  CCC=0.7408


In [ ]:
ytc_cls = np.digitize(ytc_te, bins=[2.5, 3.5], right=True)
ypc_cls = np.digitize(iso_global.transform(ypc_te), bins=[2.5, 3.5], right=True)
print("\nBalanced Acc (H/M/L):", balanced_accuracy_score(ytc_cls, ypc_cls))
print(classification_report(ytc_cls, ypc_cls,
      target_names=["Bajo","Medio","Alto"], digits=4))


Balanced Acc (H/M/L): 0.6608462252803794
              precision    recall  f1-score   support

        Bajo     0.7132    0.5193    0.6010       699
       Medio     0.5833    0.6872    0.6310       988
        Alto     0.7616    0.7760    0.7687       741

    accuracy                         0.6660      2428
   macro avg     0.6860    0.6608    0.6669      2428
weighted avg     0.6751    0.6660    0.6644      2428



In [ ]:
save_model_bundle(
    mdl, iso_global, cal_lang,
    "/content/drive/MyDrive/TP1/modelos/arousal/v2/tercero_op2",
    target_name="arousal",
    positive_lang="es",
    xgb_params=best_params,
    n_features=Xw_tr.shape[1]
)

Guardado en: /content/drive/MyDrive/TP1/modelos/arousal/v2/tercero_op2


# Modelo Final - Arousal

con **params** de "**primero**", tambien utilizado en valencia y presenta los **mejores** resultados en general

In [ ]:
best_params = {
    "objective": "reg:squarederror",
    "learning_rate": 0.05,
    "n_estimators": 2000,
    "grow_policy": "lossguide",
    "max_depth": 0,
    "max_leaves": 32,
    "min_child_weight": 10.0,
    "subsample": 0.75,
    "colsample_bytree": 0.80,
    "colsample_bynode": 0.70,
    "gamma": 0.20,
    "reg_alpha": 1.0,
    "reg_lambda": 35.0,
    "max_bin": 128,
    "tree_method": "hist",
    "random_state": 42,
    "device":"cuda",
    "n_jobs": -1,
    "early_stopping_rounds": 50,
    "eval_metric": "rmse"
}

In [ ]:
from sklearn.model_selection import train_test_split
(X_train, X_val,
 y_train, y_val,
 W_train, W_val,
 path_train, path_val,
 zrms_train, zrms_val,
 lang_train, lang_val) = train_test_split(
    Xw_tr, y_tr_w, W_full, path_tr, zrms_tr, lang_tr,
    test_size=0.15,
    random_state=42,
    shuffle=True
)

print(f"Training samples: {len(X_train)}")
print(f"Validation samples: {len(X_val)}")
print(f"Weight stats - Train: min={W_train.min():.3f}, mean={W_train.mean():.3f}, max={W_train.max():.3f}")
print(f"Weight stats - Val: min={W_val.min():.3f}, mean={W_val.mean():.3f}, max={W_val.max():.3f}")

Training samples: 112727
Validation samples: 19894
Weight stats - Train: min=0.393, mean=0.999, max=7.869
Weight stats - Val: min=0.393, mean=1.005, max=7.869


In [ ]:
mdl = train_xgb(
    X_train, y_train, W_train,
    X_val=X_val, y_val=y_val,
    params=best_params
)

[0]	validation_0-rmse:0.82369
[50]	validation_0-rmse:0.62550
[100]	validation_0-rmse:0.59817
[150]	validation_0-rmse:0.58389
[200]	validation_0-rmse:0.57351
[250]	validation_0-rmse:0.56614
[300]	validation_0-rmse:0.56003
[350]	validation_0-rmse:0.55568
[400]	validation_0-rmse:0.55172
[450]	validation_0-rmse:0.54810
[500]	validation_0-rmse:0.54520
[550]	validation_0-rmse:0.54277
[600]	validation_0-rmse:0.54021
[650]	validation_0-rmse:0.53797
[700]	validation_0-rmse:0.53622
[750]	validation_0-rmse:0.53404
[800]	validation_0-rmse:0.53211
[850]	validation_0-rmse:0.53045
[900]	validation_0-rmse:0.52895
[950]	validation_0-rmse:0.52752
[1000]	validation_0-rmse:0.52623
[1050]	validation_0-rmse:0.52484
[1100]	validation_0-rmse:0.52360
[1150]	validation_0-rmse:0.52223
[1200]	validation_0-rmse:0.52089
[1250]	validation_0-rmse:0.51981
[1300]	validation_0-rmse:0.51892
[1350]	validation_0-rmse:0.51802
[1400]	validation_0-rmse:0.51692
[1450]	validation_0-rmse:0.51585
[1500]	validation_0-rmse:0.51480


XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=0.788136481728704,
             colsample_bytree=0.6652044024893341, device='cuda',
             early_stopping_rounds=50, enable_categorical=False,
             eval_metric='rmse', feature_types=None, feature_weights=None,
             gamma=0.13549424394028625, grow_policy='lossguide',
             importance_type=None, interaction_constraints=None,
             learning_rate=0.09989730695216865, max_bin=255,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=0, max_leaves=63,
             min_child_weight=11.503343698077279, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=2000,
             n_jobs=-1, num_parallel_tree=None, ...)

In [ ]:
iso_global, cal_lang = evaluate_model(
    mdl,
    Xw_tr, y_tr_w, path_tr, zrms_tr, lang_tr,
    Xw_te, y_te_w, path_te, zrms_te, lang_te
)


=== TRAIN (clip, raw) ===  MAE=0.2748 RMSE=0.3504 CCC=0.8963
=== TEST (raw) ===  MAE=0.4665  RMSE=0.5724  CCC=0.7252
=== TEST (calibrado GLOBAL) ===  MAE=0.4611  RMSE=0.5773  CCC=0.7500
=== TEST (calibrado POR-IDIOMA) ===  MAE=0.4547  RMSE=0.5816  CCC=0.7468


In [ ]:
ytc_cls = np.digitize(ytc_te, bins=[2.5, 3.5], right=True)
ypc_cls = np.digitize(iso_global.transform(ypc_te), bins=[2.5, 3.5], right=True)
print("\nBalanced Acc (H/M/L):", balanced_accuracy_score(ytc_cls, ypc_cls))
print(classification_report(ytc_cls, ypc_cls,
      target_names=["Bajo","Medio","Alto"], digits=4))


Balanced Acc (H/M/L): 0.6472915809938624
              precision    recall  f1-score   support

        Bajo     0.6746    0.4864    0.5653       699
       Medio     0.5712    0.6700    0.6167       988
        Alto     0.7608    0.7854    0.7729       741

    accuracy                         0.6524      2428
   macro avg     0.6689    0.6473    0.6516      2428
weighted avg     0.6588    0.6524    0.6496      2428



In [ ]:
agg_kwargs = {"trim":0.10,"alpha":0.0}

In [ ]:
save_model_bundle(
    mdl, iso_global, cal_lang,
    "/content/drive/MyDrive/TP1/modelos/arousal/v2/tercero",
    target_name="arousal",
    positive_lang="es",
    xgb_params=best_params,
    n_features=Xw_tr.shape[1]
)

Guardado en: /content/drive/MyDrive/TP1/modelos/arousal/v2/tercero


## Validacion por folds

In [ ]:
groups_tr = np.array([clip_group_from_path(p) for p in path_tr])

gkf = GroupKFold(n_splits=5)
fold_stats = []

In [ ]:
import gc

In [ ]:
print("\n==== 5-Fold GroupKFold (clip-safe) on TRAIN windows ====")
for fold, (idx_tr, idx_va) in enumerate(gkf.split(Xw_tr, y_tr_w, groups=groups_tr), 1):
    X_tr, X_va = Xw_tr[idx_tr], Xw_tr[idx_va]
    y_tr, y_va = y_tr_w[idx_tr], y_tr_w[idx_va]
    w_tr      = W_full[idx_tr]
    p_tr, p_va = path_tr[idx_tr], path_tr[idx_va]
    r_tr, r_va = zrms_tr[idx_tr], zrms_tr[idx_va]
    l_tr, l_va = lang_tr[idx_tr], lang_tr[idx_va]

    model = XGBRegressor(**best_params)
    model.fit(
        X_tr, y_tr,
        sample_weight=w_tr,
        eval_set=[(X_va, y_va)],   # early stopping RMSE on val nivel ventana
        verbose=False
    )

    yp_tr = model.predict(X_tr).astype(np.float32)
    yhat_tr_clip = agg_clip(p_tr, yp_tr, r_tr)

    # true clip means (average of window labels per clip)
    ytrue_tr_clip = {}
    for p, yt in zip(p_tr, y_tr):
        g = clip_group_from_path(p)
        ytrue_tr_clip.setdefault(g, []).append(float(yt))
    ytrue_tr_clip = {g: float(np.mean(v)) for g, v in ytrue_tr_clip.items()}

    keys_tr = sorted(set(yhat_tr_clip) & set(ytrue_tr_clip))
    ypc_tr = np.array([yhat_tr_clip[k] for k in keys_tr], np.float32)
    ytc_tr = np.array([ytrue_tr_clip[k] for k in keys_tr], np.float32)

    iso = IsotonicRegression(y_min=1.0, y_max=5.0, out_of_bounds="clip").fit(ypc_tr, ytc_tr)

    # --- Validation (clip-level) ---
    yp_va = model.predict(X_va).astype(np.float32)
    yhat_va_clip = agg_clip(p_va, yp_va, r_va)

    ytrue_va_clip = {}
    for p, yt in zip(p_va, y_va):
        g = clip_group_from_path(p)
        ytrue_va_clip.setdefault(g, []).append(float(yt))
    ytrue_va_clip = {g: float(np.mean(v)) for g, v in ytrue_va_clip.items()}

    keys_va = sorted(set(yhat_va_clip) & set(ytrue_va_clip))
    ypc_va = np.array([yhat_va_clip[k] for k in keys_va], np.float32)
    ytc_va = np.array([ytrue_va_clip[k] for k in keys_va], np.float32)

    # calibrate validation clip preds
    ypc_va_cal = iso.predict(ypc_va)

    # --- metrics (clip-level) ---
    ccc  = concordance_ccc(ytc_va, ypc_va_cal)
    rmse = mean_squared_error(ytc_va, ypc_va_cal)
    mae  = mean_absolute_error(ytc_va, ypc_va_cal)

    y_true_cls = to_bins(ytc_va)
    y_pred_cls = to_bins(ypc_va_cal)
    bal_acc = balanced_accuracy_score(y_true_cls, y_pred_cls)


    print(f"Fold {fold}: CCC={ccc:.4f} | RMSE={rmse:.4f} | MAE={mae:.4f} | BalAcc={bal_acc:.4f} | best_iter={model.get_booster().best_iteration}")

    fold_stats.append({
        "ccc": ccc, "rmse": rmse, "mae": mae, "bal_acc": bal_acc,
        "best_iter": model.get_booster().best_iteration
    })
    del model, yp_tr, yp_va, yhat_tr_clip, yhat_va_clip
    gc.collect()


==== 5-Fold GroupKFold (clip-safe) on TRAIN windows ====
Fold 1: CCC=0.7731 | RMSE=0.2799 | MAE=0.4103 | BalAcc=0.6734 | best_iter=1999
Fold 2: CCC=0.7646 | RMSE=0.2949 | MAE=0.4254 | BalAcc=0.6738 | best_iter=1999
Fold 3: CCC=0.7726 | RMSE=0.2821 | MAE=0.4153 | BalAcc=0.6899 | best_iter=1995
Fold 4: CCC=0.7689 | RMSE=0.2835 | MAE=0.4173 | BalAcc=0.6841 | best_iter=1999
Fold 5: CCC=0.7651 | RMSE=0.2875 | MAE=0.4163 | BalAcc=0.6720 | best_iter=1999


In [ ]:
ccc_vals  = np.array([s["ccc"] for s in fold_stats])
rmse_vals = np.array([s["rmse"] for s in fold_stats])
mae_vals  = np.array([s["mae"] for s in fold_stats])
ba_vals   = np.array([s["bal_acc"] for s in fold_stats])
iters     = np.array([s["best_iter"] for s in fold_stats])

print(f"CCC   : mean={ccc_vals.mean():.4f}  std={ccc_vals.std(ddof=1):.4f}")
print(f"RMSE  : mean={rmse_vals.mean():.4f}  std={rmse_vals.std(ddof=1):.4f}")
print(f"MAE   : mean={mae_vals.mean():.4f}   std={mae_vals.std(ddof=1):.4f}")
print(f"BalAcc: mean={ba_vals.mean():.4f}    std={ba_vals.std(ddof=1):.4f}")
print(f"best_iteration (ES): mean={iters.mean():.1f}  min={iters.min()}  max={iters.max()}")


CCC   : mean=0.7689  std=0.0040
RMSE  : mean=0.2856  std=0.0059
MAE   : mean=0.4169   std=0.0055
BalAcc: mean=0.6786    std=0.0079
best_iteration (ES): mean=1998.2  min=1995  max=1999
